# Colab Drive RAG Run

Colab에서 RAG 실험을 돌릴 때 사용하는 노트북입니다. Drive에 데이터를 두고 공유하거나, HuggingFace embedding/reranker/LLM처럼 자원이 더 필요한 옵션을 실험할 때 사용합니다.

## 1. Drive 연결

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. 저장소 준비

In [ ]:
REPO_URL = "<repo-url>"
REPO_DIR = "/content/codeit-rag-project"
DRIVE_ROOT = "/content/drive/MyDrive/codeit_rag_project"

!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -r requirements.txt

## 3. Drive 폴더 준비

Drive에는 원본 문서, 평가 질문, 실험 결과, 백업을 분리해서 둡니다.

In [ ]:
from pathlib import Path
drive_root = Path(DRIVE_ROOT)
for subdir in ["data/raw_docs", "experiments", "backups", "reports"]:
    (drive_root / subdir).mkdir(parents=True, exist_ok=True)

print(drive_root)

## 4. RAG config 경로

Colab에서는 기존 config를 복사해서 Drive 경로로 수정하는 것을 권장합니다.

In [ ]:
from pathlib import Path
import json
import pandas as pd

RAG_CONFIG = Path("configs/experiments/rag/rag_semantic.yaml")
COLAB_CONFIG = Path("configs/experiments/rag/rag_colab_drive.yaml")
QUESTION = "예산은 얼마야?"
OUTPUT_DIR = Path(DRIVE_ROOT) / "experiments" / "rag_colab_drive"

## 5. Colab config 작성 예시

아래 셀은 예시입니다. 실제 문서와 평가 질문 파일이 Drive에 준비된 뒤 사용합니다.

In [ ]:
colab_config_text = f"""
experiment:
  name: rag_colab_drive
  author: team
  seed: 42
  contract_version: rag-v0.1

paths:
  raw_docs_dir: {DRIVE_ROOT}/data/raw_docs
  output_dir: {DRIVE_ROOT}/experiments/rag_colab_drive

artifact_policy:
  run_id:
  on_existing: overwrite

rag:
  loader:
    file_types: [txt, pdf, docx, hwpx, hwp]
  chunk:
    size: 500
    overlap: 80
    unit: char
  checkpoint:
    enabled: true
    resume: true
  embedding:
    provider: local
    model_name: hashing-char-ngram-v1
    dimension: 64
    device: auto
    normalize: true
  vector_store:
    type: memory
    path:
    collection_name: rag_colab_drive
  retriever:
    method: semantic
    top_k: 3
    score_threshold: 0.0
  reranker:
    enabled: false
    provider: huggingface
    model_name:
    top_k: 3
  answerer:
    mode: extractive
    provider: local
    model_name:
    fallback_message: 문서에서 확인하지 못했습니다.

evaluation:
  questions_path: {DRIVE_ROOT}/data/eval_questions.csv

metric:
  monitor: retrieval_hit_rate
  mode: max

backup:
  enabled: true
  on_finish: true
  on_failure: true
  backup_dir: {DRIVE_ROOT}/backups/rag_colab_drive
  include_logs: true
  include_checkpoints: true
"""
COLAB_CONFIG.write_text(colab_config_text.strip() + "\n", encoding="utf-8")
print(COLAB_CONFIG)

In [ ]:
def run(command: str) -> None:
    print(f"$ {command}")
    result = __import__("subprocess").run(command, shell=True, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(command)

def show_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

def show_csv(path, n=5):
    path = Path(path)
    return pd.read_csv(path).head(n) if path.exists() else pd.DataFrame()

## 6. 실행 전 점검

In [ ]:
run(f"python scripts/check_rag_pipeline.py --config {COLAB_CONFIG} --project-root .")

## 7. RAG 실행

In [ ]:
run(f"python scripts/run_rag_ingest.py --config {COLAB_CONFIG} --project-root .")
run(f"python scripts/run_rag_retrieve.py --config {COLAB_CONFIG} --project-root . --question '{QUESTION}'")
run(f"python scripts/run_rag_chat.py --config {COLAB_CONFIG} --project-root . --evaluate")

## 8. 결과 확인

In [ ]:
show_json(OUTPUT_DIR / "metrics.json")

In [ ]:
display(show_csv(OUTPUT_DIR / "bad_retrievals.csv"))
display(show_csv(OUTPUT_DIR / "unsupported_answers.csv"))
display(show_csv(OUTPUT_DIR / "failed_questions.csv"))

## 9. 공유할 것

- 사용한 config 경로
- Drive 결과 폴더
- `metrics.json` 요약
- 성공 질문 1개와 실패 질문 1개
- 답변과 citation 캡처